# Hyperparameter Optimization with Optuna

## Why Optuna?

Grid Search and Random Search become impractical with large search spaces.

Optuna uses adaptive optimization (Tree-structured Parzen Estimator by default) to explore promising hyperparameter combinations efficiently and has become a standard tool for tuning tabular ML models.

---

# 10.1 Install (if needed)

In [ ]:
# Uncomment if Optuna is not installed
!pip install optuna

---

# 10.2 Imports

In [ ]:
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score

---

# 10.3 Why Cross Validation?

Earlier we used a chronological train/validation split.

For **hyperparameter tuning**, there are two reasonable approaches:

1. Tune on the fixed validation split.
2. Use cross-validation inside the training set.

For a competition-style notebook, CV often gives a more stable estimate. For a production fraud system, a **time-aware validation strategy** is generally preferable because it respects the chronological nature of transactions. Since we already created a time-based holdout, we'll tune **only on the training set** and reserve the validation set for unbiased model selection.

---

# 10.4 XGBoost Objective

In [ ]:
import xgboost as xgb

def objective_xgboost(trial):

    params = {

        "objective": "binary:logistic",

        "eval_metric": "aucpr",

        "tree_method": "hist",

        "random_state": RANDOM_STATE,

        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            1000
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            15
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10,
            log=True
        ),

        "scale_pos_weight": class_weights[1]

    }

    model = xgb.XGBClassifier(**params)

    scores = cross_val_score(
        model,
        X_train_tree,
        y_train,
        scoring="average_precision",
        cv=3,
        n_jobs=-1
    )

    return scores.mean()

We optimize **Average Precision (PR-AUC)** rather than accuracy because it better reflects performance on highly imbalanced fraud datasets.

# 10.5 Run Study

In [ ]:
study_xgb = optuna.create_study(
    direction="maximize",
    study_name="XGBoost Optimization"
)

study_xgb.optimize(
    objective_xgboost,
    n_trials=30,
    show_progress_bar=True
)

In [ ]:
reset_memory()

---

# 10.6 Best Parameters

In [ ]:
print("="*60)
print("Best Trial")
print("="*60)

print(study_xgb.best_value)

print()

print(study_xgb.best_params)

---

# 10.7 Optimization History

In [ ]:
optuna.visualization.plot_optimization_history(
    study_xgb
)

---

# 10.8 Parameter Importance

In [ ]:
optuna.visualization.plot_param_importances(
    study_xgb
)

---

# 10.9 Train Final XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(

    **study_xgb.best_params,

    objective="binary:logistic",

    eval_metric="aucpr",

    random_state=RANDOM_STATE
)

In [ ]:
xgb_metrics = train_and_evaluate_model(

    model=xgb_model,

    X_train=X_train_tree,

    y_train=y_train,

    X_valid=X_valid_tree,

    y_valid=y_valid,

    feature_names=X_train_tree.columns,

    model_name="Optimized XGBoost"
)

all_results.append(xgb_metrics)